# 2026 世界盃足球賽：球員級（Player-Level）戰力評估與蒙地卡羅模擬
### 本地精緻研究專案 — 基於 EA Sports FC 26 最新球員資料庫與三大動態實戰特徵

---

## 1. 數據演進與研發心路歷程 (Data Evolution & Rationale)

在世界盃預測的探索中，我們的模型經歷了從「國家隊整體 (Team-Level)」到「球員個體 (Player-Level)」的重大演進。而在數據的選取上，更有一段克服現實瓶頸的關鍵歷程：

### 🚫 傳統聯賽統計數據的困境
在專案初期，我們嘗試使用球員在真實聯賽中的歷史表現數據（如賽季進球數、助攻數、傳球成功率等）來構建球員大名單與戰力評分。然而在實作過程中，我們遇到了兩個無法忽視的硬傷：
1. **非主流國家隊數據殘缺**：對於英格蘭、法國、阿根廷等足球大國，其隊員皆效力於五大聯賽，數據極為詳盡；但對於捷克、波赫、維德角等中游或新興國家，其大名單球員散落於全球各地二三線聯賽，面臨著嚴重的數據缺失。這迫使我們之前的版本在小組賽階段必須以「輪空判負 (0:3)」的生硬邏輯 bypass，嚴重破壞了 48 強對抗的賽程完整性。
2. **跨聯盟比較的「蘋果與橘子」問題**：不同國家的球員分佈在完全不同的聯盟與賽制中（例如英超、意甲、美職聯、沙特聯、日職聯等）。各聯賽的整體水準、防守強度、戰術風格及統計維度有巨大落差。我們無法公允地把「在日職聯進 15 球的前鋒」與「在英超進 10 球的前鋒」進行直接的數據對比。在缺乏統一基準的情況下，傳統聯賽數據很難進行跨聯盟的公允衡量。

### 💡 轉向官方遊戲數據（EA Sports FC 26 / SOFIFA）與市場身價
為了解決上述問題，我們將視角轉向了最新的 **EA Sports FC 26 (FIFA 26) 全球球員資料庫**：
- **統一且標準的評估體系**：EA Sports 擁有一支全球最龐大且深耕數十年的專業球探與數據分析師網絡。他們對全球 18,000+ 球員，使用同一套標準化的評估標準（包括整體評分 Overall、位置 Position、以及傳球、防守、速度等 110 多個屬性指標）進行全方位評分。這提供了唯一的**公允基準線**，使得我們能將效力於不同檔次聯賽的球員放在同一個標準上橫向對比。
- **身價數據反映真實影響力**：直接引入內建的球員市場身價（`value_eur`），身價是轉會市場對球員當前實力、年齡、上限和戰術價值的綜合定價，能極佳地排除單一聯賽的統計偏誤。
- **48 強全陣容實戰解鎖**：基於最新的 FC 26 乾淨數據，我們成功解鎖了 48 支參賽球隊的所有 26 人大名單，成功消除了輪空判負的代碼，達成了 100% 賽程還原與 2026 世界盃 Wiki 分組的完美還原。

---

## 2. 兩隊勝負判定與雙泊松（Double Poisson）期望模型

在足球博弈與學術研究中，**雙泊松分佈模型 (Double Poisson Model)** 是最經典且被證實極為有效的比分預測方法。我們當前版本**百分之百採用了雙泊松分析**來預測每場比賽的勝負，具體數學邏輯如下：

### 📊 (1) 進球期望值 $\lambda$ 與 $\mu$ 的推導
我們將每支球隊的 **FIFA ELO 積分 (代表歷史長期實力基礎)** 與 **球員 PQS 評分 (代表先發球員即時紙面戰力上限)** 進行動態扣減疲勞度後，計算雙方的進球期望值：
- **A 隊（主隊/實力較強者）進球期望值 $\lambda$**：
  $$\lambda = \max\left(0.2, 1.3 + \frac{\text{elo}_A - \text{elo}_B}{450} + \frac{\text{pqs}_A - \text{pqs}_B}{30} + \text{host\_boost}_A - \text{host\_boost}_B \times 0.5\right)$$
- **B 隊（客隊/挑戰者）進球期望值 $\mu$**：
  $$\mu = \max\left(0.2, 1.1 - \frac{\text{elo}_A - \text{elo}_B}{450} - \frac{\text{pqs}_A - \text{pqs}_B}{30} + \text{host\_boost}_B - \text{host\_boost}_A \times 0.5\right)$$

### 🎲 (2) 泊松隨機抽樣 (Poisson Random Sampling) 模擬比賽比分
在得出期望值 $\lambda$ 與 $\mu$ 後，我們**並非直接對比期望值的大小來分出勝負**（若直接對比，強隊將永遠獲勝，無法產生爆冷與平局）。我們是將其作為泊松分佈的參數，進行隨機數抽樣，模擬雙方在該平行宇宙中的具體進球數 $G_A$ 與 $G_B$：
$$G_A \sim \text{Poisson}(\lambda), \quad G_B \sim \text{Poisson}(\mu)$$

這代表 A 隊進 $k$ 球的機率為：$P(G_A = k) = \frac{\lambda^k e^{-\lambda}}{k!}$。這能完美還原足球競賽的「隨機性」與「低比分爆冷性」。

### 🏆 (3) 勝負判定準則
根據抽樣模擬出來的比分 $G_A$ 與 $G_B$：
1. **$G_A > G_B$**：判定 A 隊獲勝。
2. **$G_B > G_A$**：判定 B 隊獲勝。
3. **$G_A = G_B$（平局）**：
   - **小組賽階段**：直接以平局結束（DRAW），兩隊各積 1 分。
   - **淘汰賽階段**：進入 30 分鐘延長賽，再次以 $\lambda \times 0.33$ 與 $\mu \times 0.33$ 為期望值進行泊松抽樣累加進球。若依然平手，則進入門將與射手整體評分（OVR）屬性對抗的 **PK 大戰**。

此外，模型還融入了 **Dixon-Coles 修正**。由於足球比賽中 0:0 與 1:1 的平局概率通常高於獨立泊松分佈的理論值，我們在雙方均抽樣到 0 球時，有 $25\%$ 的機率將比分修正為 1:1 或維持 0:0，從而使低比分平局的預測機率更貼近現實統計。

---

## 3. 升級版三大動態實戰特徵 (Algorithm Specifications)

為使模擬更符合真實杯賽的動態競爭，我們在蒙地卡羅模擬器中引進了三個關鍵預測特徵：

### 🏠 (1) 東道主主場優勢 (Host Country Advantage)
2026 世界盃由美、加、墨三國共同舉辦。在對戰計算中，如果其中一方為東道主：
- 其期望進球數 $\lambda$ 獲得額外補貼：$\lambda_{\text{active}} = \lambda + 0.1$
- 對手的期望進球數 $\mu$ 受到客場壓制：$\mu_{\text{active}} = \mu - 0.05$

### 🔋 (2) 大賽動態疲勞度與板凳深度衰減 (Dynamic Fatigue & Bench Depth)
賽事推進中球員體力會持續流失，我們設計了滾動累加的疲勞度系統：
- 每支球隊初始疲勞值 $f = 0.0$。
- 每場常規賽（90分鐘）後累積疲勞：
  $$\Delta f = 0.04 \times (1.0 - \text{bench\_pqs})$$
  *若該場踢入延長賽 (AET)，額外累積疲勞 $+0.02$。板凳深度越深 (替補 PQS 越高)，球員輪換效率越高，疲勞度累積就越慢。*
- 計算對戰期望值時，使用受疲勞折損後的活躍指標：
  $$\text{pqs}_{\text{active}} = \text{starting\_pqs} \times (1.0 - f)$$
  $$\text{elo}_{\text{active}} = \text{fifa\_points} \times (1.0 - f \times 0.05)$$

### 🧤 (3) PK 大戰門將與射手屬性對抗 (GK OVR vs Shooters in Penalty Shootout)
若淘汰賽延長賽打平，進入點球大戰時不再是隨機的 50/50 碰運氣，而是基於球員屬性的對決：
- 提取先發陣容中 `overall` 最高者作為點球門將 ($GK_{\text{ovr}}$)，若無則預設為 60。
- 提取除門將外 `overall` 前 5 高的球員平均值，作為罰球射手實力評分 ($Shoot_{\text{ovr}}$)。
- 雙方的點球罰進機率調整為（限制在 $[0.55, 0.90]$ 區間內）：
  $$\text{rate}_A = 0.75 + \frac{Shoot_{\text{ovr}, A} - GK_{\text{ovr}, B}}{200}$$
  $$\text{rate}_B = 0.75 + \frac{Shoot_{\text{ovr}, B} - GK_{\text{ovr}, A}}{200}$$
- 依此動態機率模擬 5 輪罰球與驟死賽，決定晉級隊伍。

## 步驟 1：載入球員數據與國籍映射

我載入最新的 FC 26 球員名單，並對齊 48 支參賽隊伍的國籍拼寫。

In [ ]:
import pandas as pd
import numpy as np

df_players = pd.read_csv('FC26_20250921.csv')
print(f"最新 FC 26 球員數據庫維度: {df_players.shape}")

# 國籍映射字典
sofifa_mapping = {
    'South Korea': 'Korea Republic',
    'Ivory Coast': "Côte d'Ivoire",
    'Turkey': 'Türkiye',
    'USA': 'United States'
}

nationalities = df_players['nationality_name'].unique()
print(f"SoFIFA 中收錄的國籍總數: {len(nationalities)}")

## 步驟 2：球員級特徵工程 (Player Quality Score, PQS)

在球員級預測中，我的核心思想是計算每支球隊的：
- **先發 11 人實力 (Starting PQS)**：代表球隊的最高戰力上限。
- **替補陣容深度 (Bench PQS)**：在模擬傷病和停賽時發揮下限作用。

我挑選出每個國家的 3 個最高 Overall 的守門員，並結合另外 23 名主力位置球員，組成 26 人大名單，將其轉換為 PQS 戰力評分。

In [ ]:
# 展示前 10 名身價與戰力最豪華的國家隊
import json
with open('frontend/src/teams_db.json', 'r', encoding='utf-8') as f:
    teams_db = json.load(f)

df_teams_pqs = pd.DataFrame.from_dict(teams_db, orient='index')
display(df_teams_pqs[['team_name', 'starting_pqs', 'bench_pqs', 'market_value_million_eur']].sort_values(by='starting_pqs', ascending=False).head(10))

## 步驟 3：執行球員級 10,000 次蒙地卡羅模擬

我將球員實力 PQS 結合雙卜瓦松比分預測，對 2026 世界盃（100% 還原真實抽籤）進行 10,000 次完整賽程模擬。

以下程式碼會載入 `src/player_level_simulator.py` 並執行 10,000 次模擬，最後將結果儲存於 `src/simulation_results_player_level.csv`。

In [ ]:
import sys
sys.path.append('./src')
from player_level_simulator import run_monte_carlo

# 執行 10,000 次蒙地卡羅模擬
run_monte_carlo(10000)

## 步驟 4：模擬結果視覺化與冠軍機率排行

我載入模擬產出的 `simulation_results_player_level.csv`，展示前 15 強的奪冠機率排行，並以長條圖進行視覺化展示。

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

df_sim = pd.read_csv('./src/simulation_results_player_level.csv', index_col=0)
print("⚽ 2026 世界盃球員級模擬 - 奪冠機率前 15 強：")
display(df_sim.head(15))

# 繪製奪冠率長條圖
plt.figure(figsize=(12, 6))
top15 = df_sim.head(15)
sns.barplot(x=top15['Winner_pct'], y=top15.index, hue=top15.index, palette='viridis', legend=False)
plt.title('Top 15 Countries by 2026 World Cup Win Probability (%) - Player-Level Model (FC 26)')
plt.xlabel('Win Probability (%)')
plt.ylabel('Country')
plt.grid(axis='x', linestyle='--', alpha=0.7)
plt.show()

## 💡 額外彩蛋：啟動網頁版 Bracket 對陣圖動畫模擬器

除了在 Notebook 裡跑 10,000 次的宏觀機率分析，我還特別在 `frontend/` 目錄下架設了一個高互動性的 React 網頁。你可以一輪一輪觀看動畫對陣圖，還能隨機觸發平行宇宙事件（如球星受傷）！

### 啟動步驟：
1. 開啟終端機，切換到 `frontend` 目錄：
   ```bash
   cd frontend
   ```
2. 啟動本地開發伺服器：
   ```bash
   npm run dev
   ```
3. 打開瀏覽器訪問：`http://localhost:5173/`